# Phase 3 — Probe μ_HR : direction correcte ?

**Question** : Le signal μ_HR produit par Stage 1 a-t-il la bonne *direction* pour améliorer les métriques,
ou est-ce que la magnitude/le mécanisme est le seul problème ?

**Protocole** (zéro réentraînement) :
1. Charger Stage 1 + Stage 2 entraînés (9-node seed42)
2. Extraire `delta_pred = pred_ref - baseline - mu_HR` (UNet pur)
3. Reconstruire avec 5 variantes de μ_HR
4. Comparer F1@p99 → verdict sur l'interface AdaLN

**Variantes testées** :
- V0 `ablation` : `baseline + 0·μ_HR + delta`
- V1 `ref_1x`   : `baseline + 1·μ_HR + delta` (comportement actuel)
- V2 `scale_2x` : `baseline + 2·μ_HR + delta`
- V3 `scale_5x` : `baseline + 5·μ_HR + delta`
- V4 `oracle_α` : `baseline + α*·μ_HR + delta` (α* optimal per-sample L2)

**Décision** :
- `F1@p99(oracle_α) > 0.512` → direction correcte → AdaLN warm-start viable
- `F1@p99(scale_5x) > F1@p99(ref_1x)` → magnitude issue → AdaLN
- `F1@p99(oracle_α) ≤ 0.512` → direction fausse → problème plus profond
- `F1@p99(ablation) > F1@p99(ref_1x)` → μ_HR nuit → investiguer biais Stage 1

In [ ]:
# === Cell 1 : Bootstrap ===
import os, sys, shlex, subprocess
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/climate_data')
    REPO_DIR   = Path('/content/climate_data')
    GIT_URL    = 'https://github.com/leonelkenfack/stcdgm.git'
    GIT_BRANCH = 'four-node-causal'
    if not (REPO_DIR / '.git').exists():
        subprocess.run(shlex.split(f'git clone --depth 200 -b {GIT_BRANCH} {GIT_URL} {REPO_DIR}'), check=True)
    else:
        subprocess.run(shlex.split(f'git -C {REPO_DIR} fetch --depth=200 origin {GIT_BRANCH}'), check=True)
        subprocess.run(shlex.split(f'git -C {REPO_DIR} reset --hard origin/{GIT_BRANCH}'), check=True)
    os.chdir(str(REPO_DIR))
    sys.path.insert(0, str(REPO_DIR / 'src'))
    try:
        import torch_geometric; import cftime; import h5netcdf
        import xbatcher; import diffusers; from omegaconf import OmegaConf
    except ImportError as _e:
        print(f'Installing deps : {_e}')
        _deps = [
            'torch_geometric', 'omegaconf==2.3.0', 'hydra-core==1.3.2',
            'diffusers==0.36.0', 'einops', 'scipy', 'h5py', 'netCDF4',
            'xarray', 'dask', 'zarr', 'safetensors==0.7.0',
            'xbatcher', 'webdataset', 'cftime', 'h5netcdf',
        ]
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + _deps, check=True)
        print('Deps OK.')
else:
    DRIVE_ROOT = Path('c:/Users/reall/Desktop/climate_data')
    REPO_DIR   = DRIVE_ROOT
    sys.path.insert(0, str(REPO_DIR / 'src'))

print(f'DRIVE_ROOT : {DRIVE_ROOT}')
print(f'sys.path[0]: {sys.path[0]}')

In [ ]:
# === Cell 2 : Imports + paths + constantes ===
import json, time, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED       = 42
N_STEPS    = 18      # EDM Heun steps — identique à l'éval officielle
K_SAMPLES  = 1       # samples par batch (rapide, probe seulement)
BATCH_SIZE = 4
CFG_SCALE  = 1.0
torch.manual_seed(SEED); np.random.seed(SEED)

# Checkpoints — 9-node en priorité, 6-node en fallback
CKPT_9N = DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'epoch_last.pth'
CKPT_6N = DRIVE_ROOT / 'oracle_full'  / 'seed_42' / 'epoch_last.pth'
CKPT    = CKPT_9N if CKPT_9N.exists() else CKPT_6N
N_NODES = 9 if CKPT_9N.exists() else 6
print(f'Device  : {DEVICE}')
print(f'Ckpt    : {CKPT}  (N_NODES={N_NODES})')
print(f'Ckpt OK : {CKPT.exists()}')

# Références métriques
REF = {
    'current_9n' : {'rmse': 0.14135, 'pearson': 0.7667, 'f1_p99': 0.4531},
    'noncausal_v4': {'rmse': 0.1243,  'pearson': 0.834,  'f1_p99': 0.512 },
}

# Output
OUT_DIR = DRIVE_ROOT / 'oracle_9node' / 'seed_42' / 'phase3_probe'
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f'Output  : {OUT_DIR}')

In [ ]:
# === Cell 3 : Charger Stage 1 + Stage 2 depuis checkpoint ===
from omegaconf import OmegaConf
from st_cdgm.config import CONFIG
from st_cdgm.models.intelligible_encoder import IntelligibleEncoder
from st_cdgm.models.causal_rcn import RCNCell
from st_cdgm.models.regression_head import GraphToGridDecoder
from st_cdgm.models.diffusion_decoder import CausalDiffusionDecoder
from st_cdgm.models.edm_preconditioner import EDMConfig
from st_cdgm.training.two_stage import freeze_stage1

# Override N_NODES si 9-node
cfg = OmegaConf.to_container(CONFIG, resolve=True)
_n_nodes = N_NODES

def _load_sd(model, sd, name):
    if sd is None:
        print(f'  ⚠ {name}: state_dict absent')
        return
    res = model.load_state_dict(sd, strict=False)
    print(f'  {name}: missing={len(res.missing_keys)} unexpected={len(res.unexpected_keys)}')

print(f'Loading checkpoint  ({CKPT.name}) ...')
ck = torch.load(CKPT, map_location=DEVICE, weights_only=False)
print(f'  Keys : {[k for k in ck.keys() if not k.startswith("A_dag")][:10]}')

# ---- Stage 1 ----
encoder = IntelligibleEncoder(
    n_nodes=_n_nodes,
    n_input_features=int(CONFIG.model.n_input_features),
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
    n_layers=int(CONFIG.model.encoder_n_layers),
).to(DEVICE)

rcn_cell = RCNCell(
    n_nodes=_n_nodes,
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
).to(DEVICE)

regression_head = GraphToGridDecoder(
    n_nodes=_n_nodes,
    hidden_dim=int(CONFIG.model.encoder_hidden_dim),
    out_spatial=(int(CONFIG.data.target_h), int(CONFIG.data.target_w)),
).to(DEVICE)

_load_sd(encoder,          ck.get('encoder_state_dict'),          'encoder')
_load_sd(rcn_cell,         ck.get('rcn_cell_state_dict'),         'rcn_cell')
_load_sd(regression_head,  ck.get('regression_head_state_dict'),  'regression_head')

freeze_stage1(encoder, rcn_cell, regression_head)
encoder.eval(); rcn_cell.eval(); regression_head.eval()

# A_dag
A_dag = ck.get('A_dag')
if A_dag is None and 'rcn_cell_state_dict' in ck:
    A_dag = ck['rcn_cell_state_dict'].get('A_dag')
if A_dag is not None:
    A_dag = A_dag.to(DEVICE)
    print(f'  A_dag : shape={tuple(A_dag.shape)}  norm={A_dag.norm():.4f}')

# ---- Stage 2 ----
edm_cfg = EDMConfig(
    sigma_data=float(ck.get('sigma_data', CONFIG.diffusion.get('sigma_data', 0.5))),
)
diffusion = CausalDiffusionDecoder(CONFIG, edm_cfg).to(DEVICE)
_sd2 = (ck.get('ema_state_dict') or ck.get('diffusion_ema_state_dict')
         or ck.get('diffusion_state_dict'))
_load_sd(diffusion, _sd2, 'diffusion')
for p in diffusion.parameters(): p.requires_grad_(False)
diffusion.eval()

print('\nStage 1 + Stage 2 chargés et figés.')

In [ ]:
# === Cell 4 : Dataset K9 val (2010-2011) ===
from st_cdgm.data.pipeline import NetCDFDataPipeline
from st_cdgm.evaluation.two_stage_inference import build_two_stage_inputs

val_dataset = NetCDFDataPipeline(
    config=CONFIG,
    split='val',
    as_torch=True,
).build()

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    drop_last=False,
)
print(f'Val dataset : {len(val_dataset)} samples  |  {len(val_loader)} batches')

# Peek
_b0 = next(iter(val_loader))
print('Batch keys :', list(_b0.keys()))
for k, v in _b0.items():
    if hasattr(v, 'shape'):
        print(f'  {k:25s} shape={tuple(v.shape)}')

In [ ]:
# === Cell 5 : Probe variants — cœur du notebook ===

@torch.no_grad()
def _sample(cond, mu_HR, baseline_log):
    """Wrapper robuste autour du sampler Stage 2."""
    try:
        return diffusion.sample(
            conditioning=cond, mu_HR=mu_HR, baseline_log=baseline_log,
            num_steps=N_STEPS, cfg_scale=CFG_SCALE,
            scheduler_type='edm_karras',
        )
    except TypeError:
        # Fallback si la signature diffère
        return diffusion.sample(
            conditioning=cond, mu_HR=mu_HR, baseline_log=baseline_log,
            num_steps=N_STEPS,
        )

# Accumulateurs
acc = {k: [] for k in ['ablation', 'ref_1x', 'scale_2x', 'scale_5x', 'oracle_alpha']}
all_HR_true, all_masks = [], []
all_mu_HR, all_baseline = [], []
alpha_opts_list = []

t0 = time.time()
for i, batch in enumerate(val_loader):
    # Build inputs via helper (ou fallback manuel)
    try:
        cond, mu_HR_b, baseline_b, HR_true_b = build_two_stage_inputs(
            batch, encoder, rcn_cell, regression_head,
            config=CONFIG, device=DEVICE,
        )
    except Exception:
        # Fallback manuel : batch doit avoir les clés standard
        mu_HR_b    = batch.get('mu_HR',      batch.get('mu_hr')).to(DEVICE)
        baseline_b = batch.get('baseline_log', batch.get('baseline')).to(DEVICE)
        HR_true_b  = batch.get('HR',           batch.get('hr')).to(DEVICE)
        cond       = batch.get('LR',           batch.get('lr')).to(DEVICE)

    mask_b = batch.get('valid_mask', torch.ones_like(HR_true_b)).to(DEVICE)

    # Référence : pred avec mu_HR réel
    pred_ref = _sample(cond, mu_HR_b, baseline_b)  # [1,1,H,W] ou [B,1,H,W]
    if pred_ref.dim() == 5:  # [K,B,C,H,W] si K_SAMPLES > 1
        pred_ref = pred_ref.mean(0)

    # Extraire le delta UNet pur : HR_pred = baseline + mu_HR + delta
    # => delta = pred_ref - baseline - mu_HR
    delta_pred = pred_ref - baseline_b - mu_HR_b

    # Cinq variantes de reconstruction
    v_ablation   = baseline_b + delta_pred
    v_ref        = baseline_b + mu_HR_b        + delta_pred
    v_2x         = baseline_b + 2.0 * mu_HR_b  + delta_pred
    v_5x         = baseline_b + 5.0 * mu_HR_b  + delta_pred

    # Oracle α* per-sample : α* = <mu_HR, HR_true - baseline - delta> / ||mu_HR||²
    residual_true = HR_true_b - baseline_b - delta_pred  # ce que mu_HR devrait expliquer
    B = mu_HR_b.shape[0]
    num   = (mu_HR_b * residual_true).view(B, -1).sum(dim=1)
    denom = mu_HR_b.view(B, -1).pow(2).sum(dim=1).clamp_min(1e-8)
    alpha_opt = (num / denom).clamp(0.0, 10.0)           # [B]
    v_oracle = baseline_b + alpha_opt.view(B,1,1,1) * mu_HR_b + delta_pred

    # Accumuler
    for name, pred in [('ablation', v_ablation), ('ref_1x', v_ref),
                        ('scale_2x', v_2x), ('scale_5x', v_5x),
                        ('oracle_alpha', v_oracle)]:
        acc[name].append(pred.cpu())

    all_HR_true.append(HR_true_b.cpu())
    all_masks.append(mask_b.cpu())
    all_mu_HR.append(mu_HR_b.cpu())
    all_baseline.append(baseline_b.cpu())
    alpha_opts_list.append(alpha_opt.cpu())

    if (i + 1) % 10 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(val_loader) - i - 1)
        print(f'  batch {i+1:3d}/{len(val_loader)}  elapsed={elapsed:.0f}s  ETA={eta:.0f}s')

# Stack
preds   = {k: torch.cat(v, dim=0) for k, v in acc.items()}
HR_true = torch.cat(all_HR_true, dim=0)
masks   = torch.cat(all_masks,   dim=0)
mu_HR_all  = torch.cat(all_mu_HR,   dim=0)
baseline_all = torch.cat(all_baseline, dim=0)
alpha_opts   = torch.cat(alpha_opts_list, dim=0)

print(f'\nInférence terminée en {time.time()-t0:.0f}s')
print(f'HR_true shape : {tuple(HR_true.shape)}')
print(f'alpha_opt  : mean={alpha_opts.mean():.3f}  std={alpha_opts.std():.3f}  '
      f'min={alpha_opts.min():.3f}  max={alpha_opts.max():.3f}')
print(f'delta_pred norme moyenne : '
      f'{(preds["ref_1x"] - baseline_all - mu_HR_all).abs().mean():.5f}')

In [ ]:
# === Cell 6 : Métriques pour chaque variante ===
from st_cdgm.evaluation import compute_f1_extremes

def _pearson(a, b, eps=1e-8):
    a_c = a - a.mean(); b_c = b - b.mean()
    return (a_c * b_c).sum().item() / (a_c.norm().item() * b_c.norm().item() + eps)

def compute_metrics(pred, true, mask):
    p = pred * mask; t = true * mask
    n = mask.sum().clamp_min(1.0)
    rmse    = ((p - t).pow(2).sum() / n).sqrt().item()
    mae     = (p - t).abs().sum().item() / n.item()
    pearson = _pearson(p.flatten(), t.flatten())
    try:
        f1 = compute_f1_extremes(pred.numpy(), true.numpy(),
                                  threshold_percentiles=[95.0, 99.0])
        f1_p95 = float(f1.get('p95', 0.))
        f1_p99 = float(f1.get('p99', 0.))
    except Exception:
        f1_p95 = f1_p99 = float('nan')
    return {'rmse': rmse, 'mae': mae, 'pearson': pearson,
            'f1_p95': f1_p95, 'f1_p99': f1_p99}

print('=== Métriques par variante ===')
print(f'{"Variant":20s}  {"RMSE":>8}  {"Pearson":>8}  {"F1@p95":>8}  {"F1@p99":>8}')
print('-' * 65)
results = {}
for name, pred in preds.items():
    m = compute_metrics(pred, HR_true, masks)
    results[name] = m
    print(f'{name:20s}  {m["rmse"]:8.5f}  {m["pearson"]:8.4f}  '
          f'{m["f1_p95"]:8.4f}  {m["f1_p99"]:8.4f}')

print('-' * 65)
print(f'{"[noncausal_v4]": 20s}  {0.1243:8.5f}  {0.8344:8.4f}  {"   —":>8}  {0.512:8.4f}')
print()
print(f'alpha_opt stats : mean={alpha_opts.mean():.3f}  std={alpha_opts.std():.3f}')

In [ ]:
# === Cell 7 : Décision automatique + save ===

f1_ref     = results['ref_1x']['f1_p99']
f1_abl     = results['ablation']['f1_p99']
f1_2x      = results['scale_2x']['f1_p99']
f1_5x      = results['scale_5x']['f1_p99']
f1_oracle  = results['oracle_alpha']['f1_p99']
F1_TARGET  = 0.512   # noncausal v4
alpha_mean = float(alpha_opts.mean().item())
alpha_std  = float(alpha_opts.std().item())

print('=' * 70)
print('VERDICT PHASE 3 — PROBE μ_HR')
print('=' * 70)
print(f'  F1@p99 ablation (0×) : {f1_abl:.4f}')
print(f'  F1@p99 ref_1x        : {f1_ref:.4f}   (actuel)')
print(f'  F1@p99 scale_2x      : {f1_2x:.4f}')
print(f'  F1@p99 scale_5x      : {f1_5x:.4f}')
print(f'  F1@p99 oracle_α      : {f1_oracle:.4f}')
print(f'  F1@p99 noncausal v4  : {F1_TARGET:.4f}   (target)')
print(f'  α* moyen             : {alpha_mean:.3f} ± {alpha_std:.3f}')
print()

# Critères de décision
oracle_beats_noncausal = f1_oracle > F1_TARGET
scaling_helps          = f1_5x > f1_ref + 0.005
mu_HR_hurts            = f1_abl > f1_ref + 0.005
direction_correct      = alpha_mean > 0.5   # α* positif et raisonnable

if mu_HR_hurts:
    verdict = 'VERDICT_D_MU_HR_HARMFUL'
    print('  ✗ VERDICT D : μ_HR NUIT au modèle')
    print('  ▶ F1@p99(ablation) > F1@p99(ref) — retirer μ_HR de la reconstruction améliore')
    print('  ▶ Stage 1 produit un μ_HR biaisé. Investiguer biais de normalisation.')
    print('  ▶ NE PAS faire AdaLN — corriger Stage 1 ou supprimer μ_HR de la reconstruction.')
elif oracle_beats_noncausal and direction_correct:
    verdict = 'VERDICT_A_DIRECTION_OK_ADALN_GO'
    print('  ✓ VERDICT A : direction μ_HR CORRECTE — AdaLN warm-start VIABLE')
    print(f'  ▶ F1@p99(oracle_α={alpha_mean:.2f}×) = {f1_oracle:.4f} > {F1_TARGET:.4f} (noncausal)')
    print('  ▶ Le signal causal est dans la bonne direction mais mal calibré en magnitude.')
    print('  ▶ AdaLN apprendrait la bonne pondération dynamique de μ_HR.')
    print('  ▶ NEXT : warm-start AdaLN sur Stage 2 existant (~10-15h).')
elif scaling_helps:
    verdict = 'VERDICT_B_SCALING_HELPS_ADALN_GO'
    print('  ◑ VERDICT B : le scaling aide — AdaLN warm-start RECOMMANDÉ')
    print(f'  ▶ F1@p99(5×) = {f1_5x:.4f} > F1@p99(1×) = {f1_ref:.4f}')
    print('  ▶ μ_HR est sous-pondéré. AdaLN apprendrait l\'échelle optimale.')
    print('  ▶ NEXT : warm-start AdaLN (~10-15h).')
else:
    verdict = 'VERDICT_C_DIRECTION_WRONG'
    print('  ✗ VERDICT C : direction μ_HR INCORRECTE ou insuffisante')
    print(f'  ▶ F1@p99(oracle_α) = {f1_oracle:.4f} ≤ {F1_TARGET:.4f} même avec α* optimal')
    print('  ▶ μ_HR n\'a pas la bonne direction pour améliorer F1@p99.')
    print('  ▶ Problème plus profond : soit Stage 1 est biaisé, soit l\'information')
    print('    causale n\'est pas dans les variables du DAG.')
    print('  ▶ Investiguer : biais Stage 1, variables manquantes, ou revoir architecture.')

# Save
save = {
    'verdict': verdict,
    'n_nodes': N_NODES,
    'seed': SEED,
    'alpha_opt': {'mean': alpha_mean, 'std': alpha_std,
                  'min': float(alpha_opts.min()), 'max': float(alpha_opts.max())},
    'f1_p99': {
        'ablation': f1_abl, 'ref_1x': f1_ref,
        'scale_2x': f1_2x, 'scale_5x': f1_5x, 'oracle_alpha': f1_oracle,
        'noncausal_v4': F1_TARGET,
    },
    'full_metrics': results,
    'decision_flags': {
        'oracle_beats_noncausal': oracle_beats_noncausal,
        'scaling_helps': scaling_helps,
        'mu_HR_hurts': mu_HR_hurts,
        'direction_correct': direction_correct,
    },
}
save_path = OUT_DIR / 'probe_metrics.json'
with open(save_path, 'w') as f:
    json.dump(save, f, indent=2)
print(f'\nSauvegardé : {save_path}')

In [ ]:
# === Cell 8 : Visualisations ===
fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# --- F1@p99 bar chart ---
names  = ['ablation\n(0×)', 'ref_1×\n(actuel)', 'scale_2×', 'scale_5×', 'oracle_α*']
f1vals = [f1_abl, f1_ref, f1_2x, f1_5x, f1_oracle]
colors = ['#aaa', '#e74c3c', '#3498db', '#2980b9', '#27ae60']
bars = axes[0].bar(names, f1vals, color=colors, edgecolor='white', width=0.6)
axes[0].axhline(F1_TARGET,  ls='--', color='orange', lw=1.8, label=f'noncausal v4 = {F1_TARGET}')
axes[0].axhline(f1_ref,     ls=':',  color='red',    lw=1.2, label=f'actuel = {f1_ref:.4f}')
for bar, val in zip(bars, f1vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, val + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=8)
axes[0].set_ylabel('F1@p99')
axes[0].set_title('F1@p99 par variante μ_HR')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, max(f1vals + [F1_TARGET]) * 1.15)

# --- Distribution alpha_opt ---
axes[1].hist(alpha_opts.numpy(), bins=40, color='#27ae60', edgecolor='white', alpha=0.85)
axes[1].axvline(alpha_mean, color='black', ls='--', lw=1.5, label=f'mean={alpha_mean:.2f}')
axes[1].axvline(1.0, color='red', ls=':', lw=1.2, label='α=1 (current)')
axes[1].set_xlabel('α* optimal per-sample')
axes[1].set_ylabel('Count')
axes[1].set_title('Distribution α* oracle\n(1.0 = magnitude actuelle)')
axes[1].legend(fontsize=9); axes[1].grid(True, alpha=0.3)

# --- RMSE bar chart ---
rmse_names = list(results.keys()) + ['noncausal_v4']
rmse_vals  = [results[k]['rmse'] for k in results] + [0.1243]
clrs2 = colors + ['orange']
axes[2].bar(rmse_names, rmse_vals, color=clrs2, edgecolor='white', width=0.6)
axes[2].set_xticklabels(['abl', 'ref_1×', '2×', '5×', 'oracle', 'noncausal'],
                         rotation=30, ha='right', fontsize=9)
axes[2].set_ylabel('RMSE')
axes[2].set_title('RMSE par variante')
axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Phase 3 Probe μ_HR — {verdict}', fontsize=11, y=1.01)
plt.tight_layout()
fig_path = OUT_DIR / 'probe_viz.png'
plt.savefig(fig_path, dpi=130, bbox_inches='tight')
plt.show()
print(f'Figure : {fig_path}')